# OpenScience Project: Equatorial Pacific SST Analysis and its Influence in Peruvian Coast.

High resolution MUR SST dataset at AWS available at: https://registry.opendata.aws/mur/

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import os
import coiled

warnings.simplefilter('ignore') # filter some warning messages
xr.set_options(display_style="html")  #display dataset nicely 

## 1. Cluster: Dask

In [ ]:
%%time

cluster_type='Coiled'

if cluster_type == 'Coiled': 
    cluster=coiled.Cluster(
        region='us-west-2',
        arm=True,
        worker_vm_types=['t4g.large'],
        worker_options={"nthreads": 2},
        n_workers= 40,
        name='Vsazonova-us-west-2-large',
        wait_for_workers=False,
        compute_purchase_option='spot_with_fallback',
        software='protocoast-notebook-arm',
        workspace='esip-lab',
    )

In [ ]:
client = cluster.get_client()
client

## 2. Equatorial Pacific SST Analysis

### Initializing the data

In [ ]:
%%time

ds_sst = xr.open_zarr('https://mur-sst.s3.us-west-2.amazonaws.com/zarr-v1',consolidated=True)
ds_sst

In [ ]:
sst = ds_sst['analysed_sst']

cond = (ds_sst.mask==1) & ((ds_sst.sea_ice_fraction<.15) | np.isnan(ds_sst.sea_ice_fraction))
sst_masked = ds_sst['analysed_sst'].where(cond)
sst_masked

In [ ]:
sst_elnino = sst_masked.sel(lon=slice(-180,-70), lat=slice(-25,25)) - 273.15 # Slicing EP region for memory safety  

### Diagnostics: monthly mean, anomalies 

In [ ]:
%%time
#create a daily climatology and anomaly
climatology_mean = sst_elnino.groupby('time.dayofyear').mean('time',keep_attrs=True,skipna=False)
sst_anomaly = sst_elnino.groupby('time.dayofyear')-climatology_mean  #take out annual mean to remove trends

#create a monthly dataset, climatology, and anomaly
sst_monthly = sst_elnino.resample(time='1MS').mean('time',keep_attrs=True,skipna=False)
climatology_mean_monthly = sst_monthly.groupby('time.month').mean('time',keep_attrs=True,skipna=False)
sst_anomaly_monthly = sst_monthly.groupby('time.month')-climatology_mean_monthly  #take out annual mean to remove trends

sst_anomaly

### Visualization

In [ ]:
import hvplot.xarray
import holoviews as hv
from holoviews.operation.datashader import regrid
hv.extension('bokeh')

### Plot the daily, monthly SST timeseries. SST daily and monthly anomalies. 

we are making a time-series for the Peruvian coast, where we can assess the actual influence of the El Niño event on an example of CHL   

In [ ]:
%%time

daily = sst_elnino.sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()
monthly = sst_monthly.sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()

In [ ]:
daily.hvplot(grid=True, xlabel='Time', ylabel='Departure from the mean', title='Daily, Monthly SST Time-series, 2014-2020') * monthly.hvplot(grid=True)

In [ ]:
%%time

daily1 = sst_anomaly.drop('dayofyear').sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()
monthly1 = sst_anomaly_monthly.drop('month').sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()

In [ ]:
daily1.hvplot(grid=True, xlabel='Time', ylabel='Departure from the mean', title='Daily, Monthly SST Anomalies, 2014-2020') * monthly1.hvplot(grid=True)

### Spatial Visualization

In [ ]:
import cartopy.crs as ccrs
import geoviews.feature as gf

In [ ]:
subset = sst_elnino.sel(time='2016-01-01T09')

plot = subset.hvplot.quadmesh(
    x='lon', 
    y='lat', 
    rasterize=True,    
    geo=True, 
    cmap='turbo', 
    frame_width=400
)

basemap = gf.land() * gf.coastline() * gf.borders()
plot * basemap

In [ ]:
#sst_elnino = sst.sel(lon=slice(-180,-70), lat=slice(-25,25))

In [ ]:
%%time

sst_jan2016 = sst_elnino.sel(time=slice('2016-01-01','2016-02-01')).mean(dim='time')
sst_jan2018 = sst_elnino.sel(time=slice('2018-01-01','2018-02-01')).mean(dim='time')

In [ ]:
%%time

sst_diff = (sst_jan2016 - sst_jan2018).load()

In [ ]:
sst_diff.hvplot.quadmesh(x='lon', y='lat', 
                         geo=True,                 
                         rasterize=True, 
                         cmap='rainbow', 
                         tiles='EsriImagery', title='SST Anomalies: Equatorial Pacific')

In [ ]:
%%time

sst_dy = sst_elnino.sel(time='2016-01-01T09').load()

sst_dy.hvplot.quadmesh(x='lon', y='lat', 
                       geo=True,         
                       rasterize=True, 
                       cmap='rainbow', 
                       projection=ccrs.Orthographic(-145, 35),
                       coastline='110m', title = 'SST Distribution 01-01-2016')

## 3. Chlorophyll experiment

In [ ]:
import copernicusmarine
import hvplot.xarray

In [ ]:
# turn of some annoying warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
dataset_id = 'c3s_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M'
ds = copernicusmarine.open_dataset(dataset_id)

In [ ]:
ds

In [ ]:
%%time
da = ds['CHL'].sel(time='2016-01-01 16:00', method='nearest').load()

In [ ]:
chl_elnino = da.sel(longitude =slice(-180,-70), latitude =slice(-25,25))

In [ ]:
chl_elnino.hvplot(x='longitude', y='latitude', rasterize=True, geo=True, cmap='YlGn', clim = (0,15), tiles='OSM', title='Chlorophyll Concentration (mg/m3)')

In [ ]:
client.close()
cluster.close()